In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [10]:
!pip install gensim

In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
import math
import os
from gensim.models import FastText # ADDED: Gensim for training custom FastText

# ==========================================
# 1. CUSTOM TOKENIZER & VOCABULARY
# ==========================================
class CustomFastTextTokenizer:
    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.vocab_size = 2
        
    def clean_text(self, text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        return text.split()

    def train_and_build_matrix(self, texts, d_model=128):
        print("Tokenizing corpus for FastText...")
        sentences = [self.clean_text(text) for text in texts]
        
        print(f"Training custom FastText model on {len(sentences)} sequences...")
        # Train FastText. min_count=1 ensures EVERY word gets a vector. 
        # FastText's n-gram feature will handle any weird variations.
        ft_model = FastText(sentences=sentences, vector_size=d_model, window=5, min_count=1, workers=4, epochs=15)
        
        # Build vocabulary from the trained model
        words = list(ft_model.wv.index_to_key)
        
        # Initialize embedding matrix (vocab_size + 2 for PAD and UNK)
        self.vocab_size = len(words) + 2
        embedding_matrix = np.zeros((self.vocab_size, d_model))
        
        # UNK token gets random initialization, PAD stays 0
        embedding_matrix[1] = np.random.normal(scale=0.1, size=(d_model,))
        
        print("Transferring weights to PyTorch embedding matrix...")
        for i, word in enumerate(words):
            idx = i + 2 # Shift by 2 because 0=PAD, 1=UNK
            self.word2idx[word] = idx
            self.idx2word[idx] = word
            embedding_matrix[idx] = ft_model.wv[word]
            
        print(f"Custom FastText Vocabulary built with {self.vocab_size} tokens.")
        return torch.tensor(embedding_matrix, dtype=torch.float32)

    def encode(self, text, max_len):
        words = self.clean_text(text)
        tokens = [self.word2idx.get(w, 1) for w in words] # 1 is <UNK>
        
        # Truncate or Pad
        if len(tokens) > max_len:
            tokens = tokens[:max_len]
        else:
            tokens = tokens + [0] * (max_len - len(tokens)) # 0 is <PAD>
        return tokens

# ==========================================
# 2. DATASET BUILDER
# ==========================================
class ScratchMCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.options = ['A', 'B', 'C', 'D', 'E']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        # We encode [Prompt + Option] for all 5 choices
        input_ids = []
        for opt in self.options:
            combined_text = prompt + " " + str(row[opt])
            encoded = self.tokenizer.encode(combined_text, self.max_len)
            input_ids.append(encoded)
            
        item = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long) # Shape: (5, max_len)
        }
        
        if not self.is_test:
            ans_idx = self.options.index(row['answer'])
            item['label'] = torch.tensor(ans_idx, dtype=torch.long)
            
        return item

# ==========================================
# 3. MICRO-TRANSFORMER ARCHITECTURE
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (seq_len, batch_size, d_model)
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

class TinyMCQModel(nn.Module):
    def __init__(self, vocab_size, d_model=300, nhead=6, num_layers=1, dropout=0.3, pretrained_embeddings=None):
        super().__init__()
        # Load Pre-trained FastText weights if provided
        if pretrained_embeddings is not None:
            # freeze=False allows the model to fine-tune the FastText embeddings specifically for this task
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False, padding_idx=0)
        else:
            self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
            
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=512, 
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        
        # Projects the encoded sequence to a single score
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, input_ids):
        # input_ids: (batch_size, 5, seq_len)
        batch_size, num_opts, seq_len = input_ids.shape
        
        # Flatten options into batch dimension: (batch_size * 5, seq_len)
        x = input_ids.view(batch_size * num_opts, seq_len)
        
        # Embed and encode
        x = self.embedding(x) # (batch_size * 5, seq_len, d_model)
        x = self.transformer_encoder(x) # (batch_size * 5, seq_len, d_model)
        
        # Global Average Pooling (ignore padding in a real scenario, simplified here)
        x = x.mean(dim=1) # (batch_size * 5, d_model)
        
        # Score each option
        logits = self.classifier(x) # (batch_size * 5, 1)
        
        # Reshape back to (batch_size, 5)
        logits = logits.view(batch_size, num_opts)
        return logits

# ==========================================
# 4. TRAINING & INFERENCE PIPELINE
# ==========================================
def run_scratch_pipeline(train_df: pd.DataFrame, test_df: pd.DataFrame):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # 1. Build Tokenizer and Train FastText from ALL available text (Train + Test)
    tokenizer = CustomFastTextTokenizer()
    
    # Gathering corpus for unsupervised learning
    all_text = train_df['prompt'].tolist() + test_df['prompt'].tolist()
    for opt in ['A', 'B', 'C', 'D', 'E']:
        all_text.extend(train_df[opt].tolist())
        all_text.extend(test_df[opt].tolist())
        
    # We use d_model=128 because the dataset is small. 300 dimensions might lead to overfitting 
    # unless you have hundreds of thousands of rows.
    D_MODEL = 128 
    pretrained_embeddings = tokenizer.train_and_build_matrix(all_text, d_model=D_MODEL)

    # 2. Datasets & DataLoaders
    max_len = 128
    train_ds = ScratchMCQDataset(train_df, tokenizer, max_len=max_len)
    test_ds = ScratchMCQDataset(test_df, tokenizer, max_len=max_len, is_test=True)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    # 3. Initialize Model, Loss, Optimizer
    model = TinyMCQModel(
        vocab_size=tokenizer.vocab_size, 
        d_model=D_MODEL, # Must match FastText dimension
        nhead=4,         # 128 is divisible by 4
        num_layers=1, 
        dropout=0.3,
        pretrained_embeddings=pretrained_embeddings
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)

    # 4. Training Loop
    epochs = 25 # Increased to 25 based on established performance peak
    print(f"\nStarting Training for {epochs} Epochs...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch in train_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            logits = model(inputs) # (batch_size, 5)
            
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
        acc = correct / total
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Accuracy: {acc:.4f}")

    # 5. Inference
    print("\nGenerating Test Predictions...")
    model.eval()
    all_preds = []
    options = ['A', 'B', 'C', 'D', 'E']
    
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch['input_ids'].to(device)
            logits = model(inputs)
            
            # Sort to get top 3 indices
            scores = logits.cpu().numpy()
            top_3_idx = np.argsort(scores, axis=1)[:, ::-1][:, :3]
            
            for row in top_3_idx:
                pred_str = " ".join([options[i] for i in row])
                all_preds.append(pred_str)

    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'Prediction': all_preds
    })
    
    return submission_df


In [13]:
sub_df = run_scratch_pipeline(train_df,test_df)

Using device: cuda
Tokenizing corpus for FastText...
Training custom FastText model on 15000 sequences...
Transferring weights to PyTorch embedding matrix...
Custom FastText Vocabulary built with 2975 tokens.

Starting Training for 25 Epochs...
Epoch 1/25 | Loss: 1.4438 | Accuracy: 0.3540
Epoch 2/25 | Loss: 1.1003 | Accuracy: 0.5155
Epoch 3/25 | Loss: 0.7256 | Accuracy: 0.7190
Epoch 4/25 | Loss: 0.4417 | Accuracy: 0.8290
Epoch 5/25 | Loss: 0.2674 | Accuracy: 0.8970
Epoch 6/25 | Loss: 0.1815 | Accuracy: 0.9285
Epoch 7/25 | Loss: 0.1581 | Accuracy: 0.9365
Epoch 8/25 | Loss: 0.1113 | Accuracy: 0.9565
Epoch 9/25 | Loss: 0.0994 | Accuracy: 0.9570
Epoch 10/25 | Loss: 0.0991 | Accuracy: 0.9615
Epoch 11/25 | Loss: 0.0858 | Accuracy: 0.9665
Epoch 12/25 | Loss: 0.0851 | Accuracy: 0.9610
Epoch 13/25 | Loss: 0.0704 | Accuracy: 0.9690
Epoch 14/25 | Loss: 0.0808 | Accuracy: 0.9655
Epoch 15/25 | Loss: 0.0561 | Accuracy: 0.9745
Epoch 16/25 | Loss: 0.0740 | Accuracy: 0.9690
Epoch 17/25 | Loss: 0.0681 |

In [7]:
sub_df.to_csv('submission.csv',index=False)

In [14]:
sub_df

,id,Prediction
0,1,A E D
1,2,B D E
2,3,B E D
3,4,E D C
4,5,C A D
...,...,...
495,496,A E D
496,497,C B A
497,498,B D A
498,499,E B C
